# Detecção de Emoções em Videos - Modo Detalhado

## Etapa 1 - Importando as bibliotecas

In [ ]:
import cv2
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow
import zipfile

cv2.__version__

In [ ]:
%tensorflow_version 2.x
import tensorflow
tensorflow.__version__ # confere a versão instalada

## Etapa 2 - Conectando com o Drive e acessando os arquivos

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# extrai os vídeos de teste que estão zipados no drive
path = "/content/gdrive/My Drive/Videos.zip"
zip_object = zipfile.ZipFile(file=path, mode="r")
zip_object.extractall("./")

## Etapa 3 - Carregando o modelo

In [ ]:
from tensorflow.keras.models import load_model

diretorio = 'gdrive/My Drive/Cursos/Deteccao_Expressoes_Faciais/' # diretorio do drive onde estão os arquivos do curso

# vamos utilizar o Modelo da Arquitetura 2 pois foi o que se saiu melhor em nossos testes
model = load_model(diretorio + "modelo_02_expressoes.h5")

## Etapa 4 - Carregando o vídeo

In [ ]:
# carrega o vídeo de teste e lê o primeiro frame só pra conferir as dimensões
arquivo_video = diretorio + "testes/video_teste06.MOV"
cap = cv2.VideoCapture(arquivo_video)

conectado, video = cap.read()
print(video.shape) # mostra as dimensões do video

## Etapa 5 - Redimensionando o tamanho (opcional)


Recomendado quando o tamanho do vídeo é muito grande.
Se o vídeo tiver a resolução muito alta então pode demorar muito o processamento.


In [ ]:
redimensionar = True
# deixe True para reduzir o tamanho do vídeo salvo caso este supere a largura máxima que vamos especificar abaixo.
# para manter o tamanho original deixe False
largura_maxima = 600  # pixels. define o tamanho da largura (máxima) do vídeo a ser salvo. a altura será proporcional e é definida nos calculos abaixo

if (redimensionar and video.shape[1]>largura_maxima):
  # precisamos deixar a largura e altura proporcionais (mantendo a proporção do vídeo original) para que a imagem não fique com aparência esticada
  proporcao = video.shape[1] / video.shape[0]
  # para isso devemos calcular a proporção (largura/altura) e usaremos esse valor para calcular a altura (com base na largura que definimos acima)
  video_largura = largura_maxima
  video_altura = int(video_largura / proporcao)
else:
  video_largura = video.shape[1]
  video_altura = video.shape[0]

## Etapa 6 - Definindo as configurações do vídeo

In [ ]:
# nome do arquivo de vídeo que será salvo
nome_arquivo = diretorio+'resultado_video_teste06.avi'

# definição do codec
fourcc = cv2.VideoWriter_fourcc(*'XVID')
fps = 24

saida_video = cv2.VideoWriter(nome_arquivo, fourcc, fps, (video_largura, video_altura))

* Mais exemplos de outras configurações com o fourcc que é possível usar: https://www.programcreek.com/python/example/89348/cv2.VideoWriter_fourcc

## Etapa 7 - Processamento do vídeo e gravação do resultado

In [ ]:
from tensorflow.keras.preprocessing.image import img_to_array

# define se deixa no modo detalhado, que exibirá no canto da tela as barras com as probabilidades de cada emoção
# esse modo é pra ser usado de preferência com vídeos onde tem apenas um rosto em foco
unica_face = True
# se deixar False então o programa vai detectar a emoção de todas as faces na imagem e não vai exibir os gráficos com as probabilidades no canto

haarcascade_faces = diretorio + 'haarcascade_frontalface_alt.xml' # arquivo haarcascade

# define os tamanhos para as fontes
fonte_pequena, fonte_media = 0.4, 0.7

fonte = cv2.FONT_HERSHEY_SIMPLEX

expressoes = ["Raiva", "Nojo", "Medo", "Feliz", "Triste", "Surpreso", "Neutro"]

while (cv2.waitKey(1) < 0):
    conectado, frame = cap.read()

    if not conectado:
        break  # se ocorreu um problema ao carregar a imagem então interrompe o programa

    t = time.time() # tempo atual, antes de iniciar (vamos utilizar para calcular quanto tempo levou para executar as operações)

    # frame_video = np.copy(frame) # faz uma copia do frame do video

    if redimensionar: # se redimensionar = True então redimensiona o frame para os novos tamanhos
      frame = cv2.resize(frame, (video_largura, video_altura))

    face_cascade = cv2.CascadeClassifier(haarcascade_faces)
    cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) # converte pra grayscale
    faces = face_cascade.detectMultiScale(cinza,scaleFactor=1.2, minNeighbors=5,minSize=(30,30))

    if len(faces) > 0:
        for (x, y, w, h) in faces:

            # se detectar mais de uma face então considera aquela que possui uma maior area na imagem
            if unica_face and len(faces) > 1:
                max_area_face = faces[0]
                for face in faces:
                    if face[2] * face[3] > max_area_face[2] * max_area_face[3]:
                        max_area_face = face
                face = max_area_face
                (x,y,w,h) = max_area_face # retorna as coordenadas e tamanhos da maior face detectada nesse frame

            frame = cv2.rectangle(frame,(x,y),(x+w,y+h+10),(255,50,50),2) # desenha retângulo ao redor da face

            roi = cinza[y:y + h, x:x + w]      # extrai apenas a região de interesse (ROI) que é onde contém o rosto
            roi = cv2.resize(roi, (48, 48))    # antes de passar pra rede neural redimensiona para o tamanho das imagens de treinamento
            roi = roi.astype("float") / 255.0  # normaliza
            roi = img_to_array(roi)            # converte para array para que a rede possa processar
            roi = np.expand_dims(roi, axis=0)  # muda o shape da array

            # faz a predição - calcula as probabilidades
            result = model.predict(roi)[0]
            print(result)
            if result is not None:
              if unica_face:
                for (index, (emotion, prob)) in enumerate(zip(expressoes, result)):
                  # nomes das emoções
                  text = "{}: {:.2f}%".format(emotion, prob * 100)
                  barra = int(prob * 150)  # calcula do tamanho da barra, com base na probabilidade
                  espaco_esquerda = 7      # é a coordenada x onde inicia a barra. define quantos pixels tem de espaçamento à esquerda das barras, pra não ficar muito no canto.
                  if barra <= espaco_esquerda:
                    barra = espaco_esquerda + 1
                  # se o tamanho da barra for menor que o espaço da esquerda então deixa a barra com 1 pixel de largura
                  # isso é feito pois estamos usando esse valor para passar por coordenada, se for menor então a barra irá crescer pra esquerda
                  cv2.rectangle(frame, (espaco_esquerda, (index * 18) + 7), (barra, (index * 18) + 18), (200, 250, 20), -1)
                  cv2.putText(frame, text, (15, (index * 18) + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.25, (0, 0, 0), 1, cv2.LINE_AA)

              resultado = np.argmax(result) # encontra a emoção com maior probabilidade

              cv2.putText(frame,expressoes[resultado],(x,y-10), fonte, fonte_media,(255,255,255),1,cv2.LINE_AA) # escreve a emoção acima do rosto

            if unica_face and len(faces) > 1:
              break
            # se ja executou a face maior então não precisa percorrer as outras porque vai fazer a mesma coisa (pois vai fazer o mesmo cálculo pra ver qual é a maior)
            # então usamos break pra pular para o próximo frame

    # tempo processado = tempo atual (time.time()) - tempo inicial (t)
    cv2.putText(frame, " frame processado em {:.2f} segundos".format(time.time() - t), (20, video_altura-20), fonte, fonte_pequena, (250, 250, 250), 0, lineType=cv2.LINE_AA)

    cv2_imshow(frame)
    saida_video.write(frame) # grava o frame atual

print("Terminou")
saida_video.release()
cv2.destroyAllWindows()